# Stage 1: subscription prediction

Run top to bottom with the packages in `requirements.txt` installed.
Edit the workbook path below if needed. All model settings are fixed.

Outputs: `stage1_comparison.csv` (validation metrics) and
`stage1_predictions.csv` (all 100 test customers, ranked).


## 1. Imports and settings


In [1]:
from pathlib import Path
from importlib.metadata import version
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from qiskit import transpile
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler


In [2]:
WORKBOOK = Path("./bank_campaign_participant_release.xlsx")
COMPARISON_CSV = Path("stage1_comparison.csv")
RESULTS_CSV = Path("stage1_predictions.csv")
SEED = 42
CV_SEEDS = [42, 43, 44, 45, 46]
np.random.seed(SEED)

print("Python", sys.version.split()[0])
packages = [
    "numpy", "pandas", "scikit-learn", "scipy", "openpyxl",
    "qiskit", "qiskit-machine-learning",
]
display(pd.DataFrame({"package": packages, "version": [version(p) for p in packages]}))


Python 3.13.2


,package,version
0,numpy,2.2.6
1,pandas,2.2.3
2,scikit-learn,1.6.1
3,scipy,1.15.2
4,openpyxl,3.1.5
5,qiskit,2.3.1
6,qiskit-machine-learning,0.9.1


## 2. Read and format the workbook

Source: `pandas.read_excel`. IDs are kept for ranking; `y=yes` is the positive class.


In [3]:
tables = pd.read_excel(WORKBOOK, sheet_name=["Stage1_Train", "Stage1_Test"])
train = tables["Stage1_Train"]
test = tables["Stage1_Test"]

X = train.drop(columns=["customer_id", "y"])
y = train["y"].eq("yes").astype(int).to_numpy()
train_ids = train["customer_id"].to_numpy()
X_test = test[X.columns]


In [4]:
display(pd.DataFrame({
    "table": ["Stage1_Train", "Stage1_Test"],
    "rows": [len(train), len(test)],
    "predictors": [X.shape[1], X_test.shape[1]],
    "missing_predictor_values": [int(X.isna().sum().sum()), int(X_test.isna().sum().sum())],
}))
display(X.dtypes.astype(str).rename("dtype").to_frame())
display(train["y"].value_counts().rename_axis("label").to_frame("training_count"))


,table,rows,predictors,missing_predictor_values
0,Stage1_Train,300,15,0
1,Stage1_Test,100,15,0


,dtype
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64


,training_count
label,
no,265
yes,35


## 3. Original classical baselines

Models: `sklearn.linear_model.LogisticRegression` and
`sklearn.ensemble.GradientBoostingClassifier`.
Encoding: scikit-learn's `ColumnTransformer`, `StandardScaler`, and `OneHotEncoder`.
Both models use all original predictors, including `campaign`.


In [5]:
def make_encoder(frame, scale_numeric=True):
    numeric = frame.select_dtypes(include="number").columns.tolist()
    categorical = frame.select_dtypes(exclude="number").columns.tolist()
    return ColumnTransformer([
        ("numeric", StandardScaler() if scale_numeric else "passthrough", numeric),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])


def make_classical_models(frame):
    return {
        "logistic": make_pipeline(
            make_encoder(frame), LogisticRegression(C=1.0, max_iter=1000),
        ),
        "boosting": make_pipeline(
            make_encoder(frame, scale_numeric=False),
            GradientBoostingClassifier(
                n_estimators=100, learning_rate=0.05, max_depth=2,
                min_samples_leaf=5, random_state=SEED,
            ),
        ),
    }


Evaluation: `sklearn.model_selection.StratifiedKFold` and `sklearn.metrics`:
`roc_auc_score`, `average_precision_score`, `brier_score_loss`, `log_loss`.

Use the same three-fold splits for seeds 42–46. Probability metrics use the
300 out-of-fold predictions per seed. Precision@10 and Recall@10 use each
100-customer validation fold. Report means across seeds. All learned
preprocessing is fitted within the training folds.


In [6]:
cv_splits = {
    seed: list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y))
    for seed in CV_SEEDS
}
comparison_rows = {}
oof_predictions = {}


def predict_oof(estimator, features):
    frames = []
    for seed, splits in cv_splits.items():
        probability = np.zeros(len(y))
        fold_number = np.zeros(len(y), dtype=int)
        for fold, (fit_rows, validation_rows) in enumerate(splits, 1):
            fitted = clone(estimator)
            fitted.fit(features.iloc[fit_rows], y[fit_rows])
            probability[validation_rows] = fitted.predict_proba(features.iloc[validation_rows])[:, 1]
            fold_number[validation_rows] = fold
        frames.append(pd.DataFrame({
            "seed": seed, "fold": fold_number, "customer_id": train_ids,
            "y": y, "probability": probability,
        }))
    return pd.concat(frames, ignore_index=True)


In [7]:
def score_oof(predictions):
    seed_scores = []
    for seed, frame in predictions.groupby("seed"):
        target, probability = frame["y"], frame["probability"]
        scores = {
            "roc_auc": roc_auc_score(target, probability),
            "average_precision": average_precision_score(target, probability),
            "brier_score": brier_score_loss(target, probability),
            "log_loss": log_loss(target, probability),
        }
        precision, recall = [], []
        for fold, part in frame.groupby("fold"):
            ranked = part.sort_values(
                ["probability", "customer_id"], ascending=[False, True],
            )
            hits = int(ranked.head(10)["y"].sum())
            precision.append(hits / 10)
            recall.append(hits / int(part["y"].sum()))
        scores["precision_at_10"] = np.mean(precision)
        scores["recall_at_10"] = np.mean(recall)
        seed_scores.append(scores)
    return pd.DataFrame(seed_scores).mean().to_dict()


In [8]:
def add_comparison(name, inputs, angle_scale, predictions):
    scores = score_oof(predictions)
    comparison_rows[name] = {
        "inputs": inputs, "angle_scale": angle_scale,
        **{metric: scores[metric] for metric in [
            "roc_auc", "average_precision", "precision_at_10",
            "recall_at_10", "brier_score", "log_loss",
        ]},
    }
    table = pd.DataFrame.from_dict(comparison_rows, orient="index")
    table = table.rename_axis("model").reset_index()
    table.to_csv(COMPARISON_CSV, index=False)
    return table


In [9]:
original_models = make_classical_models(X)
for name, model in original_models.items():
    key = name + "_all"
    oof_predictions[key] = predict_oof(model, X)
    comparison = add_comparison(key, "all original", "", oof_predictions[key])
    print(key, "done")
display(comparison.round(4))


logistic_all done
boosting_all done


,model,inputs,angle_scale,roc_auc,average_precision,precision_at_10,recall_at_10,brier_score,log_loss
0,logistic_all,all original,,0.5548,0.1744,0.1533,0.1323,0.1100,0.3955
1,boosting_all,all original,,0.6099,0.2355,0.2600,0.2253,0.1025,0.3652


## 4. Classical baselines without campaign

Reuse the same builders and evaluation functions, dropping only `campaign`.


In [10]:
X_without_campaign = X.drop(columns="campaign")
reduced_models = make_classical_models(X_without_campaign)
for name, model in reduced_models.items():
    key = name + "_no_campaign"
    oof_predictions[key] = predict_oof(model, X_without_campaign)
    comparison = add_comparison(key, "no campaign", "", oof_predictions[key])
    print(key, "done")
display(comparison.round(4))


logistic_no_campaign done
boosting_no_campaign done


,model,inputs,angle_scale,roc_auc,average_precision,precision_at_10,recall_at_10,brier_score,log_loss
0,logistic_all,all original,,0.5548,0.1744,0.1533,0.1323,0.1100,0.3955
1,boosting_all,all original,,0.6099,0.2355,0.2600,0.2253,0.1025,0.3652
2,logistic_no_campaign,no campaign,,0.5591,0.1772,0.1533,0.1323,0.1096,0.3934
3,boosting_no_campaign,no campaign,,0.6141,0.2334,0.2867,0.2480,0.1027,0.3650


## 5. Four-component quantum SVM without campaign

[Qiskit's kernel tutorial](https://stfc.github.io/qiskit-machine-learning/tutorials/03_quantum_kernel.html)
describes the fidelity kernel $K(x,z)=|\langle\phi(x)|\phi(z)\rangle|^2$.
Here, [FidelityStatevectorKernel](https://stfc.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.kernels.FidelityStatevectorKernel.html)
computes it exactly, and `qiskit_machine_learning.algorithms.QSVC` fits the SVM.

We keep four PCA components, a
[ZZ feature map](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.zz_feature_map)
with two linear repetitions and alpha=2, balanced class weights, and C=1.
`sklearn.calibration.CalibratedClassifierCV` wraps the whole pipeline with
three-fold sigmoid calibration, seed 42, and ensemble=False.


In [11]:
def quantum_features(frame):
    frame = frame.drop(columns="campaign")
    frame["balance"] = np.sign(frame["balance"]) * np.log1p(abs(frame["balance"]))
    frame["previous"] = np.log1p(frame["previous"])
    # pdays=-1 means never contacted; keep that distinction before taking logs.
    frame["previously_contacted"] = (frame["pdays"] >= 0).astype(int)
    frame["pdays"] = np.log1p(frame["pdays"].clip(lower=0))
    return frame


X_quantum = quantum_features(X)
feature_map = zz_feature_map(4, reps=2, entanglement="linear", alpha=2.0)


Input range changes the encoded states and their similarities. Quantum-kernel
bandwidth studies motivate this small comparison:
[Shaydulin and Wild, 2022](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.106.042407).

`sklearn.preprocessing.MinMaxScaler` maps each training PCA component to
$[0,s\pi/2]$, with $s=1,0.5,0.25$. Scale 1 is the original $[0,\pi/2]$
range; the smaller ranges are $[0,\pi/4]$ and $[0,\pi/8]$. Later values
outside the fitted limits are clipped. PCA and scaling are learned inside
each training partition, including calibration folds.

The circuit structure and alpha stay fixed. The ZZ pair map is nonlinear,
so rescaling inputs does not uniformly multiply every gate angle. These
three ranges are our development comparison, not a replication of the paper.


In [12]:
def make_quantum_model(features, angle_scale):
    kernel = FidelityStatevectorKernel(
        feature_map=feature_map, shots=None, auto_clear_cache=False,
    )
    pipeline = Pipeline([
        ("encode", make_encoder(features)),
        ("pca", PCA(n_components=4, svd_solver="full")),
        ("angles", MinMaxScaler(
            feature_range=(0, angle_scale * np.pi / 2), clip=True,
        )),
        ("classifier", QSVC(
            quantum_kernel=kernel, C=1.0,
            class_weight="balanced", random_state=SEED,
        )),
    ])
    return CalibratedClassifierCV(
        estimator=pipeline, method="sigmoid", ensemble=False,
        cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    )


In [13]:
quantum_models = {}
for scale in [1.0, 0.5, 0.25]:
    key = f"quantum_scale_{scale:g}"
    quantum_models[scale] = make_quantum_model(X_quantum, scale)
    oof_predictions[key] = predict_oof(quantum_models[scale], X_quantum)
    comparison = add_comparison(key, "no campaign", scale, oof_predictions[key])
    print(key, "done")
display(comparison.round(4))


quantum_scale_1 done
quantum_scale_0.5 done
quantum_scale_0.25 done


,model,inputs,angle_scale,roc_auc,average_precision,precision_at_10,recall_at_10,brier_score,log_loss
0,logistic_all,all original,,0.5548,0.1744,0.1533,0.1323,0.1100,0.3955
1,boosting_all,all original,,0.6099,0.2355,0.2600,0.2253,0.1025,0.3652
2,logistic_no_campaign,no campaign,,0.5591,0.1772,0.1533,0.1323,0.1096,0.3934
3,boosting_no_campaign,no campaign,,0.6141,0.2334,0.2867,0.2480,0.1027,0.3650
4,quantum_scale_1,no campaign,1.0,0.5684,0.1488,0.1333,0.1136,0.1035,0.3597
5,quantum_scale_0.5,no campaign,0.5,0.6026,0.1784,0.1933,0.1682,0.1022,0.3554
6,quantum_scale_0.25,no campaign,0.25,0.6315,0.1993,0.1933,0.1662,0.1011,0.3509


Exact statevector simulation on CPU; no physical device or shot noise.
Circuit counts below use basis rz, sx, x, cx, optimization level 1, seed 42,
with no hardware target. The structure is identical for all three scales.


In [14]:
compiled = transpile(
    feature_map, basis_gates=["rz", "sx", "x", "cx"],
    optimization_level=1, seed_transpiler=SEED,
)
resources = []
for name, circuit in [("original", feature_map), ("transpiled", compiled)]:
    resources.append({
        "circuit": name, "qubits": circuit.num_qubits, "depth": circuit.depth(),
        "one_qubit_gates": sum(g.operation.num_qubits == 1 for g in circuit.data),
        "two_qubit_gates": sum(g.operation.num_qubits == 2 for g in circuit.data),
        "swaps": int(circuit.count_ops().get("swap", 0)),
    })
display(pd.DataFrame(resources))


,circuit,qubits,depth,one_qubit_gates,two_qubit_gates,swaps
0,original,4,19,22,12,0
1,transpiled,4,23,38,12,0


## 6. Frozen blend, test prediction, and export

Frozen 50:50 average: full-input boosting (block 3) plus quarter-scale
quantum SVM (block 5). Source: `sklearn.ensemble.VotingClassifier` (soft voting);
the fixed average is written directly below.

Score aligned out-of-fold predictions, then refit on all 300 training rows.
These are development metrics on reused customers. Test labels are hidden.


In [15]:
blend_oof = oof_predictions["boosting_all"].merge(
    oof_predictions["quantum_scale_0.25"][
        ["seed", "fold", "customer_id", "probability"]
    ],
    on=["seed", "fold", "customer_id"], suffixes=("_boosting", "_quantum"),
    validate="one_to_one",
)
blend_oof["probability"] = 0.5 * (
    blend_oof["probability_boosting"] + blend_oof["probability_quantum"]
)
comparison = add_comparison("blend_50_50", "mixed", 0.25, blend_oof)
display(comparison.round(4))


,model,inputs,angle_scale,roc_auc,average_precision,precision_at_10,recall_at_10,brier_score,log_loss
0,logistic_all,all original,,0.5548,0.1744,0.1533,0.1323,0.1100,0.3955
1,boosting_all,all original,,0.6099,0.2355,0.2600,0.2253,0.1025,0.3652
2,logistic_no_campaign,no campaign,,0.5591,0.1772,0.1533,0.1323,0.1096,0.3934
3,boosting_no_campaign,no campaign,,0.6141,0.2334,0.2867,0.2480,0.1027,0.3650
4,quantum_scale_1,no campaign,1.0,0.5684,0.1488,0.1333,0.1136,0.1035,0.3597
5,quantum_scale_0.5,no campaign,0.5,0.6026,0.1784,0.1933,0.1682,0.1022,0.3554
6,quantum_scale_0.25,no campaign,0.25,0.6315,0.1993,0.1933,0.1662,0.1011,0.3509
7,blend_50_50,mixed,0.25,0.6451,0.2552,0.2733,0.2369,0.0990,0.3454


In [16]:
final_boosting = clone(original_models["boosting"])
final_quantum = clone(quantum_models[0.25])
final_boosting.fit(X, y)
final_quantum.fit(X_quantum, y)

boosting_probability = final_boosting.predict_proba(X_test)[:, 1]
quantum_probability = final_quantum.predict_proba(quantum_features(X_test))[:, 1]
results = pd.DataFrame({
    "customer_id": test["customer_id"],
    "predicted_probability": 0.5 * (boosting_probability + quantum_probability),
})


In [17]:
results = results.sort_values(
    ["predicted_probability", "customer_id"], ascending=[False, True],
).reset_index(drop=True)
results["rank"] = np.arange(1, len(results) + 1)
results["is_top10"] = (results["rank"] <= 10).astype(int)
results.to_csv(RESULTS_CSV, index=False, float_format="%.17g")

print(f"Saved {len(comparison)} model comparisons to {COMPARISON_CSV}")
print(f"Saved {len(results)} test predictions to {RESULTS_CSV}; top ten shown below.")
display(results.head(10))


Saved 8 model comparisons to stage1_comparison.csv
Saved 100 test predictions to stage1_predictions.csv; top ten shown below.


,customer_id,predicted_probability,rank,is_top10
0,C0081,0.465190,1,1
1,C0031,0.450973,2,1
2,C0079,0.298057,3,1
3,C0170,0.243425,4,1
4,C0327,0.241362,5,1
5,C0193,0.194855,6,1
6,C0256,0.181358,7,1
7,C0349,0.178397,8,1
8,C0183,0.168263,9,1
9,C0378,0.166565,10,1
